# LeetCode 1696 — Jump Game VI bằng Python

Link đề: https://leetcode.com/problems/jump-game-vi/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Queue, Heap (Priority Queue), Monotonic Queue, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def maxResult(self, nums: List[int], k: int) -> int:
        ...
```

Lộ trình: **đệ quy thử mọi đường nhảy** (mũ) → **DP `O(n·k)`** (`dp[i] = nums[i] + max(dp[i−k..i−1])`) → nhận ra đó là bài **"max trên cửa sổ trượt"** → **heap với xóa lười** (`O(n log n)`) → **hàng đợi đơn điệu (monotonic deque)** (`O(n)`).

## 1. Mục tiêu học

1. Đặt trạng thái DP "điểm tốt nhất khi **đứng** ở ô `i`" và viết công thức chuyển từ `k` ô phía trước.
2. Nhận dạng truy vấn **max trên cửa sổ trượt độ dài `k`** ẩn trong công thức DP.
3. Cài **heap với xóa lười** (lazy deletion): chỉ bỏ phần tử khỏi đỉnh heap khi nó đã ra khỏi cửa sổ.
4. Cài **deque đơn điệu giảm** để lấy max cửa sổ trong `O(1)` khấu hao; hiểu vì sao phần tử bị "đè" không bao giờ cần lại.
5. So sánh thực nghiệm `O(n·k)`, `O(n log n)`, `O(n)` với `n = k = 10⁵`.

## 2. Đề bài & ví dụ

Cho mảng `nums` (chỉ số từ 0) và số nguyên `k`. Ban đầu bạn đứng ở ô `0`. Mỗi bước có thể nhảy **tối đa `k` ô về phía trước** mà không ra khỏi mảng: từ `i` nhảy tới bất kỳ ô nào trong `[i+1, min(n−1, i+k)]`.

Bạn muốn tới ô cuối `n−1`. **Điểm** = tổng `nums[j]` của mọi ô `j` bạn đã đứng (kể cả ô đầu và ô cuối). Trả về điểm lớn nhất.

**Ví dụ 1:** `nums = [1,-1,-2,4,-7,3], k = 2` → `7` (đi qua `[1,-1,4,3]`).

**Ví dụ 2:** `nums = [10,-5,-2,4,0,3], k = 3` → `17` (đi qua `[10,4,3]`).

**Ví dụ 3:** `nums = [1,-5,-20,4,-1,3,-6,-3], k = 2` → `0`.

**Ràng buộc:**
- `1 ≤ nums.length, k ≤ 10⁵`
- `-10⁴ ≤ nums[i] ≤ 10⁴`

In [1]:
import random
import time
import heapq
from collections import deque
from functools import lru_cache
from typing import List

examples = [
    (([1, -1, -2, 4, -7, 3], 2), 7),
    (([10, -5, -2, 4, 0, 3], 3), 17),
    (([1, -5, -20, 4, -1, 3, -6, -3], 2), 0),
]
for (nums, k), want in examples:
    print(f"nums={nums}, k={k} -> kỳ vọng {want}")

nums=[1, -1, -2, 4, -7, 3], k=2 -> kỳ vọng 7
nums=[10, -5, -2, 4, 0, 3], k=3 -> kỳ vọng 17
nums=[1, -5, -20, 4, -1, 3, -6, -3], k=2 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Quy hoạch động "tới ô i"

`dp[i]` = điểm lớn nhất của một đường đi hợp lệ từ ô 0 **dừng tại** ô `i`. Muốn đứng ở `i`, bước nhảy cuối phải xuất phát từ một ô `j ∈ [i−k, i−1]`. Vậy

`dp[i] = nums[i] + max(dp[j] for j in [max(0, i−k), i−1])`, `dp[0] = nums[0]`.

In [2]:
nums, k = [1, -1, -2, 4, -7, 3], 2
dp = [0] * len(nums)
dp[0] = nums[0]
for i in range(1, len(nums)):
    dp[i] = nums[i] + max(dp[max(0, i - k):i])
print("dp =", dp, "-> đáp án dp[-1] =", dp[-1])

dp = [1, 0, -1, 4, -3, 7] -> đáp án dp[-1] = 7


### 3.2 Max trên cửa sổ trượt

Bài toán con: với mỗi `i`, tìm `max(arr[i−k..i−1])`. Làm ngây thơ tốn `O(k)` mỗi lần. Đây là bài kinh điển **Sliding Window Maximum** (LeetCode 239).

### 3.3 `heapq` là min-heap

Muốn max-heap thì đẩy `(-giá_trị, chỉ_số)`. `h[0]` là phần tử nhỏ nhất (tức giá trị lớn nhất khi đã đổi dấu).

In [3]:
h = []
for i, v in enumerate([3, 8, 5]):
    heapq.heappush(h, (-v, i))
print("max hiện tại:", -h[0][0], "ở chỉ số", h[0][1])

max hiện tại: 8 ở chỉ số 1


### 3.4 `collections.deque`

Deque cho phép thêm/bớt **cả hai đầu** trong `O(1)`: `append`, `pop` (phải), `appendleft`, `popleft` (trái). `dq[0]`, `dq[-1]` xem hai đầu. List thường có `pop(0)` tốn `O(n)` — tránh dùng.

In [4]:
dq = deque([1, 2, 3])
dq.append(4); dq.popleft(); dq.pop()
print(dq, "| đầu =", dq[0], "| cuối =", dq[-1])

deque([2, 3]) | đầu = 2 | cuối = 3


### 3.5 Hàng đợi đơn điệu giảm

Giữ deque chứa **chỉ số**, giá trị tương ứng **giảm dần** từ trái sang phải. Khi thêm `i`: bỏ khỏi đuôi mọi chỉ số có giá trị `≤ val[i]` (chúng "già hơn và không lớn hơn" ⇒ không bao giờ là max nữa). Đầu deque luôn là max của cửa sổ; nếu đầu đã ra khỏi cửa sổ thì `popleft`.

In [5]:
def window_max(arr, k):
    dq, out = deque(), []
    for i, v in enumerate(arr):
        while dq and arr[dq[-1]] <= v:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()
        if i >= k - 1:
            out.append(arr[dq[0]])
    return out

print(window_max([1, 3, -1, -3, 5, 3, 6, 7], 3), "(LeetCode 239: [3,3,5,5,6,7])")

[3, 3, 5, 5, 6, 7] (LeetCode 239: [3,3,5,5,6,7])


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 `nums = [1,-1,-2,4,-7,3]`, `k = 2`:

| i | nums[i] | các ô có thể nhảy tới i | max dp của chúng | dp[i] |
|---|---|---|---|---|
| 0 | 1 | — | — | 1 |
| 1 | −1 | {0} | 1 | 0 |
| 2 | −2 | {0, 1} | 1 | −1 |
| 3 | 4 | {1, 2} | 0 | 4 |
| 4 | −7 | {2, 3} | 4 | −3 |
| 5 | 3 | {3, 4} | 4 | **7** |

**Trực giác tham lam sai:** "luôn nhảy tới ô lớn nhất trong tầm" — từ ô 0, tầm `{1,2}` có `−1 > −2` ⇒ nhảy tới 1; từ 1 tầm `{2,3}` ⇒ 4; từ 3 tầm `{4,5}` ⇒ 3. Tình cờ đúng ở đây, nhưng sai nói chung (mục 13). Lưu ý thêm: số **dương** thì nên ghé hết, số âm thì nhảy qua nếu được — nhưng khi `k` nhỏ có thể buộc phải dẫm lên số âm, và chọn số âm nào là bài toán DP.

## 5. Cách 1 — Brute Force / baseline: đệ quy thử mọi bước nhảy

**Ý tưởng:** `go(i)` = điểm lớn nhất từ ô `i` đi tới cuối (tính cả `nums[i]`). `go(n−1) = nums[n−1]`, `go(i) = nums[i] + max(go(j) for j in i+1..min(n−1, i+k))`. Không ghi nhớ ⇒ số đường đi mũ.

In [6]:
def max_result_brute(nums, k):
    n = len(nums)
    def go(i):
        if i == n - 1:
            return nums[i]
        return nums[i] + max(go(j) for j in range(i + 1, min(n - 1, i + k) + 1))
    return go(0)

for (nums, k), want in examples:
    print(max_result_brute(nums, k), "kỳ vọng", want)

7 kỳ vọng 7
17 kỳ vọng 17
0 kỳ vọng 0


**Độ phức tạp:** số đường đi từ 0 tới `n−1` với bước ≤ `k` tăng như dãy Fibonacci tổng quát (≈ `2^{n−1}` khi `k ≥ n`). Time mũ, Space `O(n)` stack.

**Nút thắt:** `go(j)` bị tính lại nhiều lần ⇒ ghi nhớ / DP bottom-up `O(n·k)`.

In [7]:
random.seed(0)
for n in [12, 16, 20]:
    nums = [random.randint(-10, 10) for _ in range(n)]
    t0 = time.perf_counter(); max_result_brute(nums, n)
    print(f"n={n}, k=n: {time.perf_counter() - t0:.3f}s")

n=12, k=n: 0.000s
n=16, k=n: 0.005s
n=20, k=n: 0.076s


### 5.1 Cách 1b — DP bottom-up `O(n·k)`

Đã viết ở mục 3.1. Đúng nhưng `n = k = 10⁵` cho tới `10¹⁰` phép — quá chậm.

In [8]:
def max_result_dp(nums, k):
    n = len(nums)
    dp = [0] * n
    dp[0] = nums[0]
    for i in range(1, n):
        dp[i] = nums[i] + max(dp[max(0, i - k):i])
    return dp[-1]

for (nums, k), want in examples:
    print(max_result_dp(nums, k), "kỳ vọng", want)

nums = [random.randint(-10**4, 10**4) for _ in range(3000)]
t0 = time.perf_counter(); max_result_dp(nums, 3000)
print(f"n = k = 3000: {time.perf_counter() - t0:.3f}s  (n = k = 1e5 sẽ chậm hơn ~1000 lần)")

7 kỳ vọng 7
17 kỳ vọng 17
0 kỳ vọng 0
n = k = 3000: 0.044s  (n = k = 1e5 sẽ chậm hơn ~1000 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** công thức `dp[i] = nums[i] + max(dp[i−k..i−1])` chỉ cần **max của cửa sổ trượt** độ dài `k` trên mảng `dp`. Cửa sổ chỉ **dịch sang phải** một ô mỗi bước ⇒ dùng cấu trúc hỗ trợ "thêm phần tử mới bên phải, bỏ phần tử cũ bên trái, hỏi max".

**Quy tắc thống trị (cho deque):** nếu `j1 < j2` và `dp[j1] ≤ dp[j2]` thì `j1` **không bao giờ** là lựa chọn tốt hơn `j2` cho mọi `i` sau này: `j2` vừa không tệ hơn, vừa ở trong cửa sổ lâu hơn. ⇒ loại `j1`.

**Invariant của deque:** chỉ số tăng dần, `dp` tương ứng **giảm nghiêm ngặt**; mọi chỉ số trong deque nằm trong cửa sổ `[i−k, i−1]`; đầu deque là `argmax`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP + heap với xóa lười

Từng đoạn:
1. Heap chứa `(-dp[j], j)`.
2. Tại ô `i`: trong khi đỉnh heap có chỉ số `< i − k` (ra khỏi cửa sổ) thì `heappop` — **xóa lười**: phần tử cũ chỉ bị bỏ khi nó nổi lên đỉnh.
3. `dp[i] = nums[i] − h[0][0]` (vì lưu âm).
4. Đẩy `(-dp[i], i)`.

In [9]:
def max_result_heap(nums, k):
    h = [(-nums[0], 0)]
    cur = nums[0]
    for i in range(1, len(nums)):
        while h[0][1] < i - k:
            heapq.heappop(h)
        cur = nums[i] - h[0][0]
        heapq.heappush(h, (-cur, i))
    return cur

for (nums, k), want in examples:
    print(max_result_heap(nums, k), "kỳ vọng", want)

7 kỳ vọng 7
17 kỳ vọng 17
0 kỳ vọng 0


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def maxResult(self, nums: List[int], k: int) -> int:
        h = [(-nums[0], 0)]
        cur = nums[0]
        for i in range(1, len(nums)):
            while h[0][1] < i - k:
                heapq.heappop(h)
            cur = nums[i] - h[0][0]
            heapq.heappush(h, (-cur, i))
        return cur
```

**Độ phức tạp:** mỗi chỉ số được push 1 lần, pop tối đa 1 lần ⇒ Time `O(n log n)`, Space `O(n)` (heap có thể giữ phần tử "đã chết" chưa bị pop).

### 7.2 Cách 3 — DP + hàng đợi đơn điệu (monotonic deque)

Từng đoạn:
1. `dq = deque([0])`, `dp[0] = nums[0]`.
2. Tại ô `i`: nếu `dq[0] < i − k` thì `popleft` (mỗi bước cửa sổ dịch 1 ô nên tối đa 1 phần tử ra).
3. `dp[i] = nums[i] + dp[dq[0]]`.
4. Trong khi `dp[dq[-1]] ≤ dp[i]` thì `pop` (bị `i` thống trị). Rồi `append(i)`.

In [10]:
def max_result_deque(nums, k):
    n = len(nums)
    dp = [0] * n
    dp[0] = nums[0]
    dq = deque([0])
    for i in range(1, n):
        if dq[0] < i - k:
            dq.popleft()
        dp[i] = nums[i] + dp[dq[0]]
        while dq and dp[dq[-1]] <= dp[i]:
            dq.pop()
        dq.append(i)
    return dp[-1]

for (nums, k), want in examples:
    print(max_result_deque(nums, k), "kỳ vọng", want)

7 kỳ vọng 7
17 kỳ vọng 17
0 kỳ vọng 0


**Độ phức tạp:** mỗi chỉ số vào deque 1 lần, ra tối đa 1 lần ⇒ Time `O(n)`, Space `O(n)` cho `dp` (+ `O(k)` deque).

### 7.3 Cách 4 — Deque lưu cặp `(chỉ số, dp)`, không cần mảng `dp`

Vì `dp[j]` chỉ được đọc khi `j` còn trong deque, có thể lưu luôn giá trị vào deque và bỏ mảng `dp`. Bộ nhớ phụ giảm xuống `O(k)`.

In [11]:
def max_result_deque_pairs(nums, k):
    dq = deque([(0, nums[0])])
    cur = nums[0]
    for i in range(1, len(nums)):
        if dq[0][0] < i - k:
            dq.popleft()
        cur = nums[i] + dq[0][1]
        while dq and dq[-1][1] <= cur:
            dq.pop()
        dq.append((i, cur))
    return cur

for (nums, k), want in examples:
    print(max_result_deque_pairs(nums, k), "kỳ vọng", want)

7 kỳ vọng 7
17 kỳ vọng 17
0 kỳ vọng 0


**Độ phức tạp:** Time `O(n)`, Space `O(k)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 `nums = [1,-1,-2,4,-7,3]`, `k = 2` (deque ghi `chỉ số:dp`):

| i | bỏ đầu (ra cửa sổ) | dp[i] = nums[i] + dp[đầu] | bỏ đuôi (bị thống trị) | deque sau |
|---|---|---|---|---|
| 0 | — | 1 | — | [0:1] |
| 1 | — | −1 + 1 = 0 | — | [0:1, 1:0] |
| 2 | — | −2 + 1 = −1 | — | [0:1, 1:0, 2:−1] |
| 3 | 0 | 4 + 0 = 4 | 2, 1 | [3:4] |
| 4 | — | −7 + 4 = −3 | — | [3:4, 4:−3] |
| 5 | — | 3 + 4 = 7 | 4, 3 | [5:7] |

In [12]:
def trace_deque(nums, k):
    n = len(nums); dp = [0] * n; dp[0] = nums[0]; dq = deque([0])
    print(f"i=0: dp=1, deque={[f'{j}:{dp[j]}' for j in dq]}")
    for i in range(1, n):
        out = []
        if dq[0] < i - k:
            out.append(dq.popleft())
        best_j = dq[0]
        dp[i] = nums[i] + dp[best_j]
        popped = []
        while dq and dp[dq[-1]] <= dp[i]:
            popped.append(dq.pop())
        dq.append(i)
        print(f"i={i}: ra cửa sổ={out or '-'}, dp[{i}] = {nums[i]} + dp[{best_j}]({dp[best_j]}) = {dp[i]:3d}, "
              f"bị đè={popped or '-'}, deque={[f'{j}:{dp[j]}' for j in dq]}")
    return dp[-1]

trace_deque([1, -1, -2, 4, -7, 3], 2)

i=0: dp=1, deque=['0:1']
i=1: ra cửa sổ=-, dp[1] = -1 + dp[0](1) =   0, bị đè=-, deque=['0:1', '1:0']
i=2: ra cửa sổ=-, dp[2] = -2 + dp[0](1) =  -1, bị đè=-, deque=['0:1', '1:0', '2:-1']
i=3: ra cửa sổ=[0], dp[3] = 4 + dp[1](0) =   4, bị đè=[2, 1], deque=['3:4']
i=4: ra cửa sổ=-, dp[4] = -7 + dp[3](4) =  -3, bị đè=-, deque=['3:4', '4:-3']
i=5: ra cửa sổ=-, dp[5] = 3 + dp[3](4) =   7, bị đè=[4, 3], deque=['5:7']


7

In [13]:
def trace_heap(nums, k):
    h = [(-nums[0], 0)]; cur = nums[0]
    for i in range(1, len(nums)):
        dropped = []
        while h[0][1] < i - k:
            dropped.append(heapq.heappop(h)[1])
        win_max, j = -h[0][0], h[0][1]
        cur = nums[i] + win_max
        heapq.heappush(h, (-cur, i))
        print(f"i={i}: xóa lười {dropped or '-'}, max cửa sổ = dp[{j}] = {win_max:3d}, "
              f"dp[{i}] = {cur:3d}, kích thước heap = {len(h)}")
    return cur

trace_heap([1, -5, -20, 4, -1, 3, -6, -3], 2)

i=1: xóa lười -, max cửa sổ = dp[0] =   1, dp[1] =  -4, kích thước heap = 2
i=2: xóa lười -, max cửa sổ = dp[0] =   1, dp[2] = -19, kích thước heap = 3
i=3: xóa lười [0], max cửa sổ = dp[1] =  -4, dp[3] =   0, kích thước heap = 3
i=4: xóa lười -, max cửa sổ = dp[3] =   0, dp[4] =  -1, kích thước heap = 4
i=5: xóa lười -, max cửa sổ = dp[3] =   0, dp[5] =   3, kích thước heap = 5
i=6: xóa lười -, max cửa sổ = dp[5] =   3, dp[6] =  -3, kích thước heap = 6
i=7: xóa lười -, max cửa sổ = dp[5] =   3, dp[7] =   0, kích thước heap = 7


0

Quan sát trace heap: kích thước heap tăng dần vì phần tử ngoài cửa sổ **không bị xóa ngay** — chỉ bị xóa khi chúng nằm ở đỉnh. Đó là đặc điểm của xóa lười; đổi lại code rất ngắn.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maxResult(self, nums: List[int], k: int) -> int:
        dq = deque([(0, nums[0])])          # (chỉ số, dp), dp giảm dần
        cur = nums[0]
        for i in range(1, len(nums)):
            if dq[0][0] < i - k:
                dq.popleft()
            cur = nums[i] + dq[0][1]
            while dq and dq[-1][1] <= cur:
                dq.pop()
            dq.append((i, cur))
        return cur

sol = Solution()
for (nums, k), want in examples:
    got = sol.maxResult(nums, k)
    print(got, "OK" if got == want else "SAI")

7 OK
17 OK
0 OK


## 10. Vì sao thuật toán đúng

**DP đúng:** mọi đường đi tới `i` (với `i > 0`) có bước cuối xuất phát từ một `j ∈ [i−k, i−1]`, và phần đường đi tới `j` phải tối ưu (nếu không, thay bằng đường tốt hơn tới `j` sẽ được đường tốt hơn tới `i`) ⇒ `dp[i] = nums[i] + max dp[j]`.

**Deque trả về đúng max cửa sổ.** Chứng minh invariant: *trước khi xử lý `i`, deque chứa (theo thứ tự tăng) các chỉ số `j ∈ [i−1−k, i−1]` sao cho không tồn tại `j' > j` trong khoảng đó với `dp[j'] ≥ dp[j]`*.
- Chỉ số bị bỏ ở đuôi bị một chỉ số mới hơn có `dp` không nhỏ hơn thống trị — nó không bao giờ là max duy nhất cho các `i` sau.
- Chỉ số bị bỏ ở đầu đã ra khỏi cửa sổ.
- Vì `dp` trong deque giảm dần, phần tử đầu (còn trong cửa sổ) là max của **những phần tử chưa bị thống trị**, cũng chính là max của cả cửa sổ.

**Chỉ cần một `popleft` mỗi bước** vì cửa sổ dịch đúng 1 ô và chỉ số trong deque tăng dần: chỉ có `i−k−1` có thể vừa ra.

**Heap đúng:** đỉnh heap sau khi xóa lười là phần tử lớn nhất trong số phần tử có chỉ số `≥ i−k`; mọi phần tử chỉ số `≥ i−k` đều đã được đẩy (chúng `< i`). ✓

## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `k` là bước nhảy tối đa.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đệ quy thuần | mũ | `O(n)` | chỉ `n ≤ 20` |
| 1b. DP `O(nk)` | `O(n·k)` | `O(n)` | `n = k = 10⁵` ⇒ 10¹⁰ |
| 2. Heap xóa lười | `O(n log n)` | `O(n)` | đơn giản, an toàn |
| 3. Deque + mảng dp | `O(n)` | `O(n)` | tối ưu |
| 4. Deque cặp | `O(n)` | `O(k)` | **nộp bài** |

In [15]:
random.seed(1)
n = 10**5
nums = [random.randint(-10**4, 10**4) for _ in range(n)]
for k in [1, 100, 10**5]:
    res = []
    for name, fn in [("heap", max_result_heap), ("deque", max_result_deque), ("pairs", max_result_deque_pairs)]:
        t0 = time.perf_counter(); r = fn(nums, k); res.append(r)
        print(f"k={k:6d} {name:5s}: {r:10d}  {time.perf_counter() - t0:.3f}s")
    assert len(set(res)) == 1

k=     1 heap :    1986691  0.033s
k=     1 deque:    1986691  0.014s
k=     1 pairs:    1986691  0.012s
k=   100 heap :  251458795  0.040s


k=   100 deque:  251458795  0.015s
k=   100 pairs:  251458795  0.014s


k=100000 heap :  251458795  0.054s
k=100000 deque:  251458795  0.014s
k=100000 pairs:  251458795  0.014s


## 12. Bộ test

In [16]:
fns = [max_result_dp, max_result_heap, max_result_deque, max_result_deque_pairs,
       lambda a, k: Solution().maxResult(a, k)]

for (nums, k), want in examples:
    assert max_result_brute(nums, k) == want
    for fn in fns:
        assert fn(nums, k) == want

edge = [
    (([5], 1), 5),                    # một ô
    (([-3], 7), -3),                  # một ô âm, k > n
    (([1, -100, 2], 1), -97),         # k = 1: buộc đi qua mọi ô
    (([1, -100, 2], 2), 3),           # nhảy qua ô âm
    (([-1, -1, -1, -1], 3), -2),      # nhảy thẳng từ 0 tới cuối
    (([0, 0, 0], 100000), 0),
]
for (nums, k), want in edge:
    assert max_result_brute(nums, k) == want, (nums, k)
    for fn in fns:
        assert fn(nums, k) == want, (fn, nums, k)

random.seed(2024)
for _ in range(1500):
    n = random.randint(1, 14)
    k = random.randint(1, 16)
    nums = [random.randint(-20, 20) for _ in range(n)]
    want = max_result_brute(nums, k)
    for fn in fns:
        assert fn(nums, k) == want, (nums, k)
print("ví dụ + edge + 1500 bộ ngẫu nhiên khớp đệ quy vét cạn")

ví dụ + edge + 1500 bộ ngẫu nhiên khớp đệ quy vét cạn


In [17]:
# đối chiếu các cách nhanh với DP O(nk) trên n lớn hơn, giá trị đúng ràng buộc
random.seed(77)
for _ in range(40):
    n = random.randint(100, 800); k = random.randint(1, n)
    nums = [random.randint(-10**4, 10**4) for _ in range(n)]
    want = max_result_dp(nums, k)
    assert max_result_heap(nums, k) == max_result_deque(nums, k) == max_result_deque_pairs(nums, k) == want
print("n tới 800: các cách nhanh khớp DP O(nk)")

n tới 800: các cách nhanh khớp DP O(nk)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam: nhảy tới ô có giá trị lớn nhất trong tầm

Chỉ nhìn một bước, không tính hệ quả xa hơn.

In [18]:
def wrong_greedy(nums, k):
    n = len(nums); i = 0; total = nums[0]
    while i < n - 1:
        j = max(range(i + 1, min(n - 1, i + k) + 1), key=lambda t: nums[t])
        total += nums[j]; i = j
    return total

random.seed(3)
for _ in range(5000):
    n = random.randint(2, 8); k = random.randint(1, 4)
    nums = [random.randint(-9, 9) for _ in range(n)]
    if wrong_greedy(nums, k) != max_result_brute(nums, k):
        print("phản ví dụ:", nums, "k =", k, "| tham lam =", wrong_greedy(nums, k), "| đúng =", max_result_brute(nums, k))
        break

phản ví dụ: [2, 6, 9] k = 2 | tham lam = 11 | đúng = 17


**Sửa:** DP xét mọi ô xuất phát trong cửa sổ.

### Lỗi 2 — Quên bỏ chỉ số đã ra khỏi cửa sổ ở đầu deque

(Ghi chú: dùng `<` thay cho `≤` khi bỏ đuôi vẫn đúng — chỉ giữ thêm các phần tử bằng nhau. Lỗi nguy hiểm thật sự là **quên `popleft`**.)

In [19]:
def wrong_no_popleft(nums, k):
    n = len(nums); dp = [0] * n; dp[0] = nums[0]; dq = deque([0])
    for i in range(1, n):
        dp[i] = nums[i] + dp[dq[0]]          # SAI: dq[0] có thể < i - k
        while dq and dp[dq[-1]] <= dp[i]:
            dq.pop()
        dq.append(i)
    return dp[-1]

nums, k = [10, -5, -5, -5, 1], 1
print("sai =", wrong_no_popleft(nums, k), "| đúng =", max_result_deque(nums, k))

sai = 11 | đúng = -4


Với `k = 1` phải đi qua mọi ô (tổng `−4`), nhưng bản lỗi cứ "nhảy" từ ô 0 có `dp = 10` dù ô đó ở quá xa. **Sửa:** `if dq[0] < i - k: dq.popleft()` trước khi đọc đầu deque.

### Lỗi 3 — Heap không xóa lười (lấy đỉnh mà không kiểm tra chỉ số)

In [20]:
def wrong_heap(nums, k):
    h = [(-nums[0], 0)]; cur = nums[0]
    for i in range(1, len(nums)):
        cur = nums[i] - h[0][0]              # SAI: đỉnh có thể ở ngoài cửa sổ
        heapq.heappush(h, (-cur, i))
    return cur

nums, k = [10, -5, -5, -5, 1], 2
print("sai =", wrong_heap(nums, k), "| đúng =", max_result_heap(nums, k))

sai = 11 | đúng = 6


**Sửa:** `while h[0][1] < i - k: heappop(h)`.

### Lỗi 4 — Cửa sổ lệch một: dùng `i − k + 1` hoặc `i − k − 1`

Từ `j` nhảy được tới `i` khi `i − j ≤ k` ⇔ `j ≥ i − k`. Điều kiện ra cửa sổ là `j < i − k`. Dùng `j < i − k + 1` làm cửa sổ hẹp đi một ô.

In [21]:
def wrong_off_by_one(nums, k):
    n = len(nums); dp = [0] * n; dp[0] = nums[0]; dq = deque([0])
    for i in range(1, n):
        while dq[0] < i - k + 1 and len(dq) > 1:   # SAI: hẹp 1 ô
            dq.popleft()
        dp[i] = nums[i] + dp[dq[0]]
        while dq and dp[dq[-1]] <= dp[i]:
            dq.pop()
        dq.append(i)
    return dp[-1]

nums, k = [1, -1, -2, 4, -7, 3], 2
print("ví dụ 1: sai =", wrong_off_by_one(nums, k), "| đúng =", max_result_deque(nums, k))

ví dụ 1: sai = -2 | đúng = 7


**Sửa:** điều kiện `dq[0] < i - k`. Mẹo: thử `k = 1` bằng tay (chỉ `j = i − 1` hợp lệ).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- DP dạng `dp[i] = f(i) + max/min(dp[i−k..i−1])` ⇒ **monotonic deque** (hoặc heap nếu muốn an toàn).
- Cửa sổ cố định độ dài, dịch từng ô ⇒ mỗi bước tối đa 1 `popleft`.
- Heap + xóa lười: dễ viết, chịu được cửa sổ không đều (chỉ cần "hạn chót" của mỗi phần tử).
- Kiểm tra off-by-one bằng `k = 1` (bắt buộc đi qua mọi ô ⇒ đáp án = `sum(nums)`).
- Họ bài: 239 Sliding Window Maximum, 1425 Constrained Subsequence Sum (gần như y hệt, nhưng được bắt đầu ở bất kỳ đâu), 862 Shortest Subarray with Sum at Least K.

## 15. Biến thể của bài

1. **Trả về đường đi tối ưu:** lưu `parent[i] = dq[0]` khi tính `dp[i]`, rồi truy ngược.
2. **LeetCode 1425 — Constrained Subsequence Sum:** `dp[i] = nums[i] + max(0, max dp cửa sổ)` (được bắt đầu mới), đáp án `max(dp)`.
3. **Bước nhảy trong `[a, b]`** (tối thiểu `a`, tối đa `b`): cửa sổ `[i−b, i−a]` — thêm phần tử vào deque **trễ** `a` bước.
4. **Tối thiểu hóa chi phí** thay vì tối đa: đảo dấu hoặc dùng deque tăng.
5. **Jump Game I/II** (55, 45): chỉ hỏi có tới được / ít bước nhất — tham lam là đủ.

In [22]:
def max_result_with_path(nums, k):
    n = len(nums); dp = [0] * n; dp[0] = nums[0]; par = [-1] * n; dq = deque([0])
    for i in range(1, n):
        if dq[0] < i - k:
            dq.popleft()
        par[i] = dq[0]; dp[i] = nums[i] + dp[dq[0]]
        while dq and dp[dq[-1]] <= dp[i]:
            dq.pop()
        dq.append(i)
    path, i = [], n - 1
    while i != -1:
        path.append(i); i = par[i]
    return dp[-1], path[::-1]

for (nums, k), want in examples:
    best, path = max_result_with_path(nums, k)
    print(f"điểm={best}, đường đi chỉ số={path}, giá trị={[nums[i] for i in path]}")

điểm=7, đường đi chỉ số=[0, 1, 3, 5], giá trị=[1, -1, 4, 3]
điểm=17, đường đi chỉ số=[0, 3, 4, 5], giá trị=[10, 4, 0, 3]
điểm=0, đường đi chỉ số=[0, 1, 3, 5, 7], giá trị=[1, -5, 4, 3, -3]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy | thử mọi bước | mũ | `O(n)` | kiểm chứng |
| 1b. DP thẳng | max `k` ô trước | `O(n·k)` | `O(n)` | `k` rất nhỏ |
| 2. Heap lười | max-heap + bỏ phần tử hết hạn | `O(n log n)` | `O(n)` | viết nhanh, ít sai |
| 3. Deque + dp | deque giảm | `O(n)` | `O(n)` | cần mảng dp (truy vết) |
| 4. Deque cặp | deque chứa `(i, dp)` | `O(n)` | `O(k)` | **nộp bài** |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `maxResult([2, -3, 4], 1)` và `maxResult([2, -3, 4], 2)`?

**Bài 2 (tính tay):** lập bảng `dp` và trạng thái deque cho `nums = [3, -2, -2, 5, -1]`, `k = 2`.

**Bài 3 (điền TODO):** hoàn thiện `max_result_range(nums, a, b)` — mỗi bước nhảy **ít nhất `a`, nhiều nhất `b`** ô; trả về `None` nếu không tới được cuối.

**Bài 4 (tự viết lại):** viết lại Cách 2 (heap) nhưng **không đổi dấu**: dùng `heapq` với khóa `-dp` được không? Hãy viết phiên bản dùng lớp bọc `__lt__` đảo chiều.

**Bài 5 (mini challenge):** LeetCode 1425 — *Constrained Subsequence Sum*: chọn một dãy con khác rỗng sao cho hai phần tử liên tiếp được chọn cách nhau ≤ `k` chỉ số, tối đa tổng. Viết lời giải `O(n)` và kiểm chứng bằng vét cạn.

In [23]:
def max_result_range(nums, a, b):
    # TODO: dp[i] = nums[i] + max(dp[i-b .. i-a]); thêm chỉ số i-a vào deque khi xét i
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `k = 1` ⇒ đi qua mọi ô. (2) `k = 2` ⇒ nhảy qua `−3`. (3) Đáp án `3` và `6`.

**Bài 2:** (1) `dp[0] = 3`, `dp[1] = 1`, `dp[2] = 1`. (2) `dp[3] = 5 + max(dp[1], dp[2]) = 6`. (3) `dp[4] = −1 + max(dp[2], dp[3]) = 5`.

**Bài 3:** (1) Ô `j` vào cửa sổ của `i` khi `j = i − a`. (2) Ô không tới được có `dp = −∞`; đừng đẩy vào deque. (3) Ra cửa sổ khi `j < i − b`.

**Bài 4:** (1) Định nghĩa `class Item` với `__lt__` so sánh ngược. (2) Hoặc dùng `heapq._heappush_max` (nội bộ, không khuyến khích). (3) Cách sạch nhất vẫn là đổi dấu.

**Bài 5:** (1) `dp[i] = nums[i] + max(0, max dp[i−k..i−1])`. (2) Deque giảm như bài này. (3) Đáp án `max(dp)`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert Solution().maxResult([2, -3, 4], 1) == 3 and Solution().maxResult([2, -3, 4], 2) == 6

# Bài 2
assert max_result_deque([3, -2, -2, 5, -1], 2) == 5

# Bài 3
NEG = float("-inf")
def max_result_range_ans(nums, a, b):
    n = len(nums); dp = [NEG] * n; dp[0] = nums[0]; dq = deque()
    for i in range(1, n):
        j = i - a
        if j >= 0 and dp[j] != NEG:
            while dq and dp[dq[-1]] <= dp[j]:
                dq.pop()
            dq.append(j)
        while dq and dq[0] < i - b:
            dq.popleft()
        if dq:
            dp[i] = nums[i] + dp[dq[0]]
    return None if dp[-1] == NEG else dp[-1]

def brute_range(nums, a, b):
    n = len(nums)
    @lru_cache(maxsize=None)
    def go(i):
        if i == n - 1: return nums[i]
        best = NEG
        for j in range(i + a, min(n - 1, i + b) + 1):
            best = max(best, go(j))
        return nums[i] + best
    r = go(0)
    return None if r == NEG else r

random.seed(21)
for _ in range(1000):
    n = random.randint(1, 15); a = random.randint(1, 4); b = random.randint(a, 6)
    nums = [random.randint(-10, 10) for _ in range(n)]
    assert max_result_range_ans(nums, a, b) == brute_range(nums, a, b), (nums, a, b)
    assert max_result_range_ans(nums, 1, b) == max_result_deque(nums, b)

# Bài 4
class MaxItem:
    __slots__ = ("v", "i")
    def __init__(self, v, i): self.v, self.i = v, i
    def __lt__(self, other): return self.v > other.v

def max_result_heap_cls(nums, k):
    h = [MaxItem(nums[0], 0)]; cur = nums[0]
    for i in range(1, len(nums)):
        while h[0].i < i - k:
            heapq.heappop(h)
        cur = nums[i] + h[0].v
        heapq.heappush(h, MaxItem(cur, i))
    return cur

for (nums, k), want in examples:
    assert max_result_heap_cls(nums, k) == want
print("Bài 1-4: OK")

Bài 1-4: OK


In [25]:
# Bài 5 — LeetCode 1425 Constrained Subsequence Sum
def constrained_subset_sum(nums, k):
    dq = deque(); best = None; dp = [0] * len(nums)
    for i, x in enumerate(nums):
        if dq and dq[0] < i - k:
            dq.popleft()
        dp[i] = x + max(0, dp[dq[0]] if dq else 0)
        while dq and dp[dq[-1]] <= dp[i]:
            dq.pop()
        dq.append(i)
        best = dp[i] if best is None else max(best, dp[i])
    return best

def brute_1425(nums, k):
    n = len(nums); best = None
    for mask in range(1, 1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        if all(idx[t + 1] - idx[t] <= k for t in range(len(idx) - 1)):
            s = sum(nums[i] for i in idx)
            best = s if best is None else max(best, s)
    return best

assert constrained_subset_sum([10, 2, -10, 5, 20], 2) == 37
assert constrained_subset_sum([-1, -2, -3], 1) == -1
assert constrained_subset_sum([10, -2, -10, -5, 20], 2) == 23
random.seed(22)
for _ in range(400):
    nums = [random.randint(-10, 10) for _ in range(random.randint(1, 11))]
    k = random.randint(1, 5)
    assert constrained_subset_sum(nums, k) == brute_1425(nums, k)
print("Bài 5: 1425 khớp vét cạn")

Bài 5: 1425 khớp vét cạn


## 20. Template nhớ nhanh + Checklist

```python
# dp[i] = nums[i] + max(dp[i-k .. i-1])  bằng deque giảm dần
dq = deque([(0, nums[0])]); cur = nums[0]
for i in range(1, n):
    if dq[0][0] < i - k:            # 1) bỏ phần tử ra khỏi cửa sổ
        dq.popleft()
    cur = nums[i] + dq[0][1]        # 2) đầu deque = max cửa sổ
    while dq and dq[-1][1] <= cur:  # 3) bỏ phần tử bị thống trị
        dq.pop()
    dq.append((i, cur))             # 4) thêm phần tử mới
return cur
```

**Checklist trước khi submit:**
- [ ] `dp[0] = nums[0]` (ô đầu luôn được tính).
- [ ] Điều kiện ra cửa sổ: `j < i − k` (thử `k = 1`).
- [ ] Bỏ đầu **trước** khi đọc max; thêm `i` **sau** khi tính `dp[i]`.
- [ ] Không dùng `list.pop(0)` (chậm) — dùng `deque`.
- [ ] Thử `n = 1`, `k ≥ n`, mảng toàn âm.